<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 32 - Sistemas de Gestión de Almacenamiento

**Módulo:** Sistemas de Big Data (SBD - 5074)

---

## Objetivo

En este notebook compararemos **formatos de archivo** comunes en Big Data:
- **CSV**: formato de texto basado en filas
- **JSON**: formato semi-estructurado
- **Parquet**: formato binario columnar

Mediremos el **tamaño en disco**, el **tiempo de escritura** y el **tiempo de lectura** para entender cuándo usar cada uno.

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import json

np.random.seed(42)
print("Librerías cargadas correctamente ✅")

---

## 1. Generación del dataset sintético

Creamos un dataset de ventas con **500.000 registros** que simula datos reales de una empresa.

In [ ]:
N = 500_000

categorias = ['Electrónica', 'Ropa', 'Alimentación', 'Hogar', 'Deportes']
regiones = ['Norte', 'Sur', 'Este', 'Oeste', 'Centro']
metodos_pago = ['Tarjeta', 'Efectivo', 'Transferencia', 'PayPal']

df = pd.DataFrame({
    'id_venta': range(1, N + 1),
    'fecha': pd.date_range('2024-01-01', periods=N, freq='min'),
    'producto': np.random.choice(categorias, N),
    'region': np.random.choice(regiones, N),
    'cantidad': np.random.randint(1, 50, N),
    'precio_unitario': np.round(np.random.uniform(5.0, 500.0, N), 2),
    'descuento': np.round(np.random.uniform(0, 0.3, N), 2),
    'metodo_pago': np.random.choice(metodos_pago, N),
    'id_cliente': np.random.randint(1000, 50000, N),
    'satisfaccion': np.random.randint(1, 6, N)
})

df['total'] = np.round(df['cantidad'] * df['precio_unitario'] * (1 - df['descuento']), 2)

print(f"Dataset generado: {len(df):,} registros x {len(df.columns)} columnas")
print(f"Memoria en RAM: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print()
df.head()

---

## 2. Escritura en diferentes formatos

Vamos a guardar el mismo dataset en **CSV**, **JSON** y **Parquet**, midiendo el tiempo de cada operación.

In [ ]:
os.makedirs('datos_formatos', exist_ok=True)

resultados = {}

# --- CSV ---
inicio = time.time()
df.to_csv('datos_formatos/ventas.csv', index=False)
t_csv_write = time.time() - inicio

# --- JSON ---
inicio = time.time()
df.to_json('datos_formatos/ventas.json', orient='records', lines=True, date_format='iso')
t_json_write = time.time() - inicio

# --- Parquet ---
inicio = time.time()
df.to_parquet('datos_formatos/ventas.parquet', index=False)
t_parquet_write = time.time() - inicio

size_csv = os.path.getsize('datos_formatos/ventas.csv') / 1024**2
size_json = os.path.getsize('datos_formatos/ventas.json') / 1024**2
size_parquet = os.path.getsize('datos_formatos/ventas.parquet') / 1024**2

print("=" * 55)
print("RESULTADOS DE ESCRITURA")
print("=" * 55)
print(f"{'Formato':<12} {'Tiempo (s)':<15} {'Tamaño (MB)':<15}")
print("-" * 55)
print(f"{'CSV':<12} {t_csv_write:<15.3f} {size_csv:<15.2f}")
print(f"{'JSON':<12} {t_json_write:<15.3f} {size_json:<15.2f}")
print(f"{'Parquet':<12} {t_parquet_write:<15.3f} {size_parquet:<15.2f}")
print()
print(f"💡 Parquet es {size_csv/size_parquet:.1f}x más pequeño que CSV")
print(f"💡 Parquet es {size_json/size_parquet:.1f}x más pequeño que JSON")

---

## 3. Lectura y rendimiento de consultas

Ahora leemos cada formato y medimos cuánto tarda. También medimos una consulta de filtrado típica.

In [ ]:
def medir_lectura(funcion_lectura, ruta, n_repeticiones=3):
    """Mide el tiempo promedio de lectura."""
    tiempos = []
    for _ in range(n_repeticiones):
        inicio = time.time()
        df_temp = funcion_lectura(ruta)
        tiempos.append(time.time() - inicio)
    return np.mean(tiempos), df_temp

t_csv_read, _ = medir_lectura(pd.read_csv, 'datos_formatos/ventas.csv')
t_json_read, _ = medir_lectura(
    lambda r: pd.read_json(r, lines=True), 'datos_formatos/ventas.json'
)
t_parquet_read, _ = medir_lectura(pd.read_parquet, 'datos_formatos/ventas.parquet')

print("=" * 55)
print("RESULTADOS DE LECTURA (promedio de 3 ejecuciones)")
print("=" * 55)
print(f"{'Formato':<12} {'Tiempo lectura (s)':<20}")
print("-" * 55)
print(f"{'CSV':<12} {t_csv_read:<20.4f}")
print(f"{'JSON':<12} {t_json_read:<20.4f}")
print(f"{'Parquet':<12} {t_parquet_read:<20.4f}")
print()
print(f"💡 Parquet es {t_csv_read/t_parquet_read:.1f}x más rápido que CSV en lectura")

---

## 4. Ventaja columnar: lectura selectiva de columnas

Una de las mayores ventajas de Parquet es poder leer **solo las columnas necesarias**.

In [ ]:
columnas_necesarias = ['producto', 'total']

# CSV: tiene que leer TODO el archivo y luego filtrar
inicio = time.time()
df_csv = pd.read_csv('datos_formatos/ventas.csv', usecols=columnas_necesarias)
t_csv_cols = time.time() - inicio

# Parquet: solo lee las columnas pedidas del disco
inicio = time.time()
df_parquet = pd.read_parquet('datos_formatos/ventas.parquet', columns=columnas_necesarias)
t_parquet_cols = time.time() - inicio

print("=" * 55)
print(f"LECTURA SELECTIVA: solo columnas {columnas_necesarias}")
print("=" * 55)
print(f"CSV (usecols):      {t_csv_cols:.4f} s")
print(f"Parquet (columns):  {t_parquet_cols:.4f} s")
print(f"\n💡 Parquet es {t_csv_cols/t_parquet_cols:.1f}x más rápido leyendo columnas selectivas")
print("   Esto se debe a que el formato columnar solo lee los bytes de las columnas pedidas")

---

## 5. Visualización comparativa

Resumimos todos los resultados en gráficos para facilitar la comparación.

In [ ]:
import matplotlib.pyplot as plt

formatos = ['CSV', 'JSON', 'Parquet']
tamanos = [size_csv, size_json, size_parquet]
tiempos_escritura = [t_csv_write, t_json_write, t_parquet_write]
tiempos_lectura = [t_csv_read, t_json_read, t_parquet_read]

colores = ['#3498db', '#e67e22', '#2ecc71']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Tamaño en disco
bars1 = axes[0].bar(formatos, tamanos, color=colores, edgecolor='black', linewidth=0.5)
axes[0].set_title('Tamaño en disco (MB)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('MB')
for bar, val in zip(bars1, tamanos):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}', ha='center', fontweight='bold')

# Tiempo de escritura
bars2 = axes[1].bar(formatos, tiempos_escritura, color=colores, edgecolor='black', linewidth=0.5)
axes[1].set_title('Tiempo de escritura (s)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Segundos')
for bar, val in zip(bars2, tiempos_escritura):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', fontweight='bold')

# Tiempo de lectura
bars3 = axes[2].bar(formatos, tiempos_lectura, color=colores, edgecolor='black', linewidth=0.5)
axes[2].set_title('Tiempo de lectura (s)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Segundos')
for bar, val in zip(bars3, tiempos_lectura):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', fontweight='bold')

plt.suptitle('Comparativa de formatos de archivo - 500K registros', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Escalabilidad: ¿cómo cambian los resultados con más datos?

Vamos a repetir las mediciones con datasets de diferentes tamaños para observar cómo escala cada formato.

In [ ]:
tamanos_test = [10_000, 50_000, 100_000, 250_000, 500_000]
resultados_escala = {'n': [], 'csv_size': [], 'json_size': [], 'parquet_size': [],
                     'csv_read': [], 'json_read': [], 'parquet_read': []}

for n in tamanos_test:
    df_test = df.head(n).copy()
    resultados_escala['n'].append(n)

    df_test.to_csv('datos_formatos/test.csv', index=False)
    df_test.to_json('datos_formatos/test.json', orient='records', lines=True)
    df_test.to_parquet('datos_formatos/test.parquet', index=False)

    resultados_escala['csv_size'].append(os.path.getsize('datos_formatos/test.csv') / 1024**2)
    resultados_escala['json_size'].append(os.path.getsize('datos_formatos/test.json') / 1024**2)
    resultados_escala['parquet_size'].append(os.path.getsize('datos_formatos/test.parquet') / 1024**2)

    inicio = time.time()
    pd.read_csv('datos_formatos/test.csv')
    resultados_escala['csv_read'].append(time.time() - inicio)

    inicio = time.time()
    pd.read_json('datos_formatos/test.json', lines=True)
    resultados_escala['json_read'].append(time.time() - inicio)

    inicio = time.time()
    pd.read_parquet('datos_formatos/test.parquet')
    resultados_escala['parquet_read'].append(time.time() - inicio)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(resultados_escala['n'], resultados_escala['csv_size'], 'o-', label='CSV', color='#3498db', linewidth=2)
ax1.plot(resultados_escala['n'], resultados_escala['json_size'], 's-', label='JSON', color='#e67e22', linewidth=2)
ax1.plot(resultados_escala['n'], resultados_escala['parquet_size'], '^-', label='Parquet', color='#2ecc71', linewidth=2)
ax1.set_xlabel('Número de registros')
ax1.set_ylabel('Tamaño (MB)')
ax1.set_title('Tamaño en disco vs N registros', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(resultados_escala['n'], resultados_escala['csv_read'], 'o-', label='CSV', color='#3498db', linewidth=2)
ax2.plot(resultados_escala['n'], resultados_escala['json_read'], 's-', label='JSON', color='#e67e22', linewidth=2)
ax2.plot(resultados_escala['n'], resultados_escala['parquet_read'], '^-', label='Parquet', color='#2ecc71', linewidth=2)
ax2.set_xlabel('Número de registros')
ax2.set_ylabel('Tiempo (s)')
ax2.set_title('Tiempo de lectura vs N registros', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Escalabilidad de formatos de archivo', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("📊 Observa cómo Parquet escala mucho mejor que CSV y JSON")
print("   tanto en tamaño como en velocidad de lectura.")

---

## 7. Resumen y conclusiones

| Formato | Tamaño | Lectura | Escritura | Mejor para |
|---------|--------|---------|-----------|------------|
| **CSV** | Grande | Lento | Rápido | Intercambio simple, importación/exportación |
| **JSON** | Muy grande | Muy lento | Medio | APIs, datos semi-estructurados, configuración |
| **Parquet** | Pequeño | Muy rápido | Medio | Análisis, data lakes, consultas analíticas |

### Regla general para Big Data:
- **Ingestión** → CSV o JSON (por compatibilidad)
- **Almacenamiento a largo plazo** → Parquet o ORC (tamaño y rendimiento)
- **Consultas analíticas** → Parquet (lectura columnar selectiva)

---

## Ejercicios para practicar

1. Repite el experimento con un dataset de 1 millón de registros. ¿Cómo cambian las proporciones?
2. Investiga cómo se comporta Parquet con compresión `gzip` vs `snappy`.
3. ¿Qué pasa si el dataset tiene muchas columnas de texto largo? ¿Cambia la ventaja de Parquet?